# Huberman Lab Youtube Extraction

### Setup

In [1]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time
import pandas as pd
import os

In [2]:
# Get playlist page using Selenium since the content is dynamically loaded
driver = webdriver.Chrome()  # Make sure you have ChromeDriver installed
driver.get("https://www.youtube.com/playlist?list=PLPNW_gerXa4Pc8S2qoUQc5e8Ir97RLuVW")

# Function to scroll to bottom of page
def scroll_to_bottom():
    last_height = driver.execute_script("return document.documentElement.scrollHeight")
    while True:
        # Scroll down
        driver.execute_script("window.scrollTo(0, document.documentElement.scrollHeight);")
        # Wait for new videos to load
        time.sleep(2)
        # Calculate new scroll height
        new_height = driver.execute_script("return document.documentElement.scrollHeight")
        if new_height == last_height:
            break
        last_height = new_height

# Scroll to load all videos
scroll_to_bottom()

# Wait for the video elements to load
wait = WebDriverWait(driver, 30)
video_elements = wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, "ytd-playlist-video-renderer")))

# Extract video information
video_data = []
for element in video_elements:
    # Get URL and title
    video_link = element.find_element(By.CSS_SELECTOR, "a#video-title").get_attribute('href')
    title = element.find_element(By.CSS_SELECTOR, "a#video-title").text
    
    if video_link and '/watch?v=' in video_link:
        video_data.append({
            'url': video_link,
            'title': title
        })

driver.quit()

# Create DataFrame
df_videos = pd.DataFrame(video_data)
df_videos.head()


,url,title
0,https://www.youtube.com/watch?v=SsKkZTjUJEk&li...,How to Grow From Doing Hard Things | Michael E...
1,https://www.youtube.com/watch?v=JaRGJVrJBQ8&li...,The Science & Practice of Perfecting Your Slee...
2,https://www.youtube.com/watch?v=2Y_PxTxLFVg&li...,Improving Science & Restoring Trust in Public ...
3,https://www.youtube.com/watch?v=2ZkJtIouKzE&li...,Improving Health With Stronger Brain-Body Conn...
4,https://www.youtube.com/watch?v=inUkNZe5H3k&li...,Healing From Grief & Loss | Dr. Mary-Frances O...


In [3]:
# Extract video key from URL by splitting on '=' and taking everything after it
df_videos['video_key'] = df_videos['url'].str.split('=').str[1].str.split('&').str[0]
df_videos.head()



,url,title,video_key
0,https://www.youtube.com/watch?v=SsKkZTjUJEk&li...,How to Grow From Doing Hard Things | Michael E...,SsKkZTjUJEk
1,https://www.youtube.com/watch?v=JaRGJVrJBQ8&li...,The Science & Practice of Perfecting Your Slee...,JaRGJVrJBQ8
2,https://www.youtube.com/watch?v=2Y_PxTxLFVg&li...,Improving Science & Restoring Trust in Public ...,2Y_PxTxLFVg
3,https://www.youtube.com/watch?v=2ZkJtIouKzE&li...,Improving Health With Stronger Brain-Body Conn...,2ZkJtIouKzE
4,https://www.youtube.com/watch?v=inUkNZe5H3k&li...,Healing From Grief & Loss | Dr. Mary-Frances O...,inUkNZe5H3k


In [4]:
df_videos.reset_index(inplace=True)
df_videos.sort_values('index', ascending=False, inplace=True)
df_videos.reset_index(drop=True, inplace=True)
df_videos.drop(columns=['index'], inplace=True)
df_videos.reset_index(inplace=True)
df_videos.rename(columns={'index': 'id'}, inplace=True)
df_videos.head()

,id,url,title,video_key
0,0,https://www.youtube.com/watch?v=4b6bwcWK6GE&li...,Welcome to the Huberman Lab Podcast,4b6bwcWK6GE
1,1,https://www.youtube.com/watch?v=H-XfCl-HpRM&li...,How Your Brain Works & Changes,H-XfCl-HpRM
2,2,https://www.youtube.com/watch?v=nm1TxQj9IsQ&li...,Master Your Sleep & Be More Alert When Awake,nm1TxQj9IsQ
3,3,https://www.youtube.com/watch?v=nwSkFq4tyC0&li...,"Using Science to Optimize Sleep, Learning & Me...",nwSkFq4tyC0
4,4,https://www.youtube.com/watch?v=NAATB55oxeQ&li...,"How to Defeat Jet Lag, Shift Work & Sleeplessness",NAATB55oxeQ


In [5]:
os.makedirs('data', exist_ok=True)
df_videos.to_csv('data/huberman_videos.csv', index=False)

In [10]:
[print(x) for x in df_videos.iloc[:2]['url']]

https://www.youtube.com/watch?v=4b6bwcWK6GE&list=PLPNW_gerXa4Pc8S2qoUQc5e8Ir97RLuVW&index=297&pp=iAQB0gcJCdQJAYcqIYzv
https://www.youtube.com/watch?v=H-XfCl-HpRM&list=PLPNW_gerXa4Pc8S2qoUQc5e8Ir97RLuVW&index=296&pp=iAQB


[None, None]